# 온톨로지 실습

**Ontology · 개념 체계 · OWL**

분야의 개념과 개념 사이의 관계를 형식을 갖춰 정의한 체계. 사람과 프로그램이 같은 뜻으로 데이터를 읽게 만든다.

소재 분야에서 이해하기: 소재·공정·물성의 관계를 정의해 서로 다른 실험실의 데이터를 같은 항목으로 합친다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [W3C OWL 웹 온톨로지 언어](https://www.w3.org/OWL/)

## 1. 개념과 관계를 형식으로 적어보기

온톨로지의 최소 형태는 "무엇이 무엇의 하위 개념인가"와 "각 개념이 무엇을 가져야 하는가"입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 개념 계층: 자식 -> 부모
SUBCLASS = {
    'Material': None,
    'Metal': 'Material', 'Ceramic': 'Material', 'Polymer': 'Material',
    'Alloy': 'Metal', 'SteelAlloy': 'Alloy', 'TitaniumAlloy': 'Alloy',
    'Property': None, 'MechanicalProperty': 'Property', 'Hardness': 'MechanicalProperty',
    'VickersHardness': 'Hardness', 'BandGap': 'Property',
}

# 각 개념의 인스턴스가 반드시 가져야 하는 항목
REQUIRED = {
    'Material': ['id'],
    'Alloy': ['baseElement'],
    'VickersHardness': ['value', 'unit', 'load_kgf'],
}

def ancestors(concept):
    chain = []
    parent = SUBCLASS.get(concept)
    while parent:
        chain.append(parent)
        parent = SUBCLASS.get(parent)
    return chain

for concept in ('SteelAlloy', 'VickersHardness', 'BandGap'):
    print('%-16s 상위 개념: %s' % (concept, ' > '.join(ancestors(concept)) or '(없음)'))

## 2. 상위 개념으로 물어보기

계층이 있으면 "합금인 시료"를 물었을 때 강 합금과 티타늄 합금이 모두 나옵니다.

In [ ]:
samples = [
    {'id': 'A1', 'type': 'SteelAlloy', 'baseElement': 'Fe'},
    {'id': 'A2', 'type': 'SteelAlloy', 'baseElement': 'Fe'},
    {'id': 'B1', 'type': 'TitaniumAlloy', 'baseElement': 'Ti'},
    {'id': 'C1', 'type': 'Ceramic'},
]

def is_a(concept, target):
    return concept == target or target in ancestors(concept)

for target in ('Alloy', 'Metal', 'Material', 'Ceramic'):
    hits = [s['id'] for s in samples if is_a(s['type'], target)]
    print('%-10s 에 해당하는 시료: %s' % (target, hits))
print('\n하위 개념을 일일이 나열하지 않아도 상위 개념 한 번으로 질의가 됩니다.')

## 3. 제약 위반 찾기

온톨로지는 어떤 기록이 불완전한지도 기계적으로 알려줍니다.

In [ ]:
def required_for(concept):
    fields = []
    for level in [concept] + ancestors(concept):
        fields += REQUIRED.get(level, [])
    return list(dict.fromkeys(fields))

def check(record):
    missing = [field for field in required_for(record['type']) if field not in record]
    return missing

for record in samples:
    missing = check(record)
    print('%-4s (%-15s) %s' % (record['id'], record['type'],
                               '통과' if not missing else '누락 ' + ', '.join(missing)))

measurement = {'type': 'VickersHardness', 'value': 431, 'unit': 'HV'}
print('\n측정 기록 검사:', check(measurement) or '통과')
print('load_kgf 가 없으면 같은 HV 값이라도 비교할 수 없으므로 필수 항목입니다.')

## 4. 해석

온톨로지는 데이터를 더 많이 담기 위한 것이 아니라, **같은 뜻으로 읽히게** 하기 위한 것입니다.
실무에서는 소재 분야 표준 온톨로지(EMMO 등)를 재사용하고 부족한 개념만 확장합니다.
OWL·RDF로 표현하면 `rdflib` 같은 도구로 추론과 검증을 자동화할 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#ontology)을 여세요.